# Data Cleaning

This notebook is for table :
- director
- cast_member
- movie_director
- movie_cast
- reviews


Code by Roland Vonck

# Setup

In [73]:
import pandas as pd
import numpy as np

# Source files, loaded once
meta = pd.read_csv("meta.csv", encoding="utf-8")
movie_lookup = pd.read_csv("movie_lookup.csv")
expert_r = pd.read_csv("expert_r_extract.csv")
user_r = pd.read_csv("user_r_extract.csv")

# Repairs garbled characters (GÃ©rard -> Gérard); used for cast and directors
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

# Check the excel columns and data types
print(meta.columns)
meta.info()

C:\Users\rolan\AppData\Local\Temp\ipykernel_10660\3675283961.py:8: DtypeWarning: Columns (0: idvscore) have mixed types. Specify dtype option on import or set low_memory=False.
  user_r = pd.read_csv("user_r_extract.csv")


Index(['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director',
       'genre', 'summary', 'awards', 'metascore', 'userscore', 'RelDate'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   url        11364 non-null  str    
 1   title      11364 non-null  str    
 2   studio     11014 non-null  str    
 3   rating     10297 non-null  str    
 4   runtime    11109 non-null  float64
 5   cast       7662 non-null   str    
 6   director   11350 non-null  str    
 7   genre      11344 non-null  str    
 8   summary    5467 non-null   str    
 9   awards     4387 non-null   str    
 10  metascore  11364 non-null  int64  
 11  userscore  9259 non-null   float64
 12  RelDate    11364 non-null  str    
dtypes: float64(2), int64(1), str(10)
memory usage: 4.3 MB


# director

In [74]:
# fix garbled names where needed
meta['director'] = meta['director'].dropna().apply(fix_encoding)

In [75]:
# split into one director per row and clean
director_long = meta['director'].dropna().str.split(',').explode().str.strip()

# drop empty strings (trailing commas) and leftover suffix fragments like "Jr."
director_long = director_long[~director_long.isin(['', 'Jr.', 'Sr.', 'II', 'III'])]
director_long.head(5)


0    Lynn Hershman-Leeson
1        Dan Trachtenberg
2         Brad Silberling
3            Jamie Linden
4         Cameron Cairnes
Name: director, dtype: str

In [76]:
# build table, sort A-Z, assign IDs
director = (
    director_long
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .to_frame(name='directorname')
    .reset_index(drop=True)
)
director.insert(0, 'directorid', director.index + 1)
director.head(5)

,directorid,directorname
0,1,A. Dean Bell
1,2,A.B. Shawky
2,3,A.J. Eaton
3,4,A.J. Edwards
4,5,Aaron Aites


In [77]:
# check for odd characters like before in Cast
print(director.shape)
print(director[director['directorname'].str.contains('Ã|Â', na=False)])  # should be empty
director.head(20)

(6136, 2)
Empty DataFrame
Columns: [directorid, directorname]
Index: []


,directorid,directorname
0,1,A. Dean Bell
1,2,A.B. Shawky
2,3,A.J. Eaton
3,4,A.J. Edwards
4,5,Aaron Aites
5,6,Aaron B. Koontz
6,7,Aaron Blaise
7,8,Aaron Brookner
8,9,Aaron Fisher
9,10,Aaron Harvey


In [78]:
# check for other weird characters
odd = sorted({c for name in director['directorname'] for c in name if ord(c) > 127})
print(odd)

['Á', 'Å', 'É', 'Ô', 'Ö', 'Ø', 'à', 'á', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ø', 'ú', 'û', 'ü']


In [79]:
# export
director.to_csv("director_extract.csv", index=False)

# cast_member

In [80]:
# fix garbled names where needed

meta['cast'] = meta['cast'].dropna().apply(fix_encoding)

In [81]:
# split into one actor per row and clean
cast_long = meta['cast'].dropna().str.split(',').explode().str.strip()

# drop empty strings (trailing commas) and leftover suffix fragments like "Jr."
cast_long = cast_long[~cast_long.isin(['', 'Jr.', 'Sr.', 'II', 'III'])]
cast_long.head(5)


1         John Gallagher Jr.
1               John Goodman
1    Mary Elizabeth Winstead
2                 Jonah Hill
2             Morgan Freeman
Name: cast, dtype: str

In [82]:
# build table, sort A-Z, assign IDs
cast_member = (
    cast_long
    .replace({'Zsolt PÃ': 'Zsolt Páll'})
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .to_frame(name='castname')
    .reset_index(drop=True)
)
cast_member.insert(0, 'castid', cast_member.index + 1)
cast_member.head(5)

,castid,castname
0,1,50 Cent
1,2,6ix9ine
2,3,A Martinez
3,4,A. Brian Daniels
4,5,A. Patrice Moreau


In [83]:
# check and export
print(cast_member.shape)
print(cast_member[cast_member['castname'].str.contains('Ã|Â', na=False)])  # should be empty
cast_member.head(20)

cast_member.to_csv("cast_member_extract.csv", index=False)

(24861, 2)
Empty DataFrame
Columns: [castid, castname]
Index: []


In [84]:
# check for more weird symbols
odd = sorted({c for name in cast_member['castname'] for c in name if ord(c) > 127})
print(odd)

['À', 'Á', 'Å', 'Æ', 'Ç', 'É', 'Í', 'Ó', 'Ô', 'Ö', 'Ø', 'Ú', 'Ü', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'ú', 'û', 'ü', 'ý', 'þ', 'ň', 'ź', 'Ž']


In [85]:
# no other weird characters, let's find Zsolt 
hits = meta[meta['cast'].str.contains('Zsolt P', na=False)][['title', 'url', 'cast']]
print(len(hits))
hits

1


,title,url,cast
4212,In Fabric,https://www.metacritic.com/movie/in-fabric,"Barry Adamson,Caroline Catz,Deborah Griffin,Fa..."


Zsolt PÃ was found, and fixed with .replace({'Zsolt PÃ': 'Zsolt Páll'}) in the code of "# build table, sort A-Z, assign IDs" so the IDs are regenerated with the correct name in its correct alphabetical spot

# movie_director

In [86]:
# loading the movie -> director strings, and the new director table with IDs
junction = movie_lookup
director = pd.read_csv("director_extract.csv")

# keeping only movieid + director, dropping movies without director info
director_data = junction[['movieid', 'director']].dropna().copy()

# repairing encoding, then split "A,B" into a list
director_data['director'] = director_data['director'].apply(fix_encoding)
director_data['director'] = director_data['director'].str.split(',')
print(director_data)

       movieid                     director
0            1                   [Don Hahn]
1            2             [Hayao Miyazaki]
2            3                [Jay Russell]
3            4                 [Steve Carr]
4            5       [Francis Ford Coppola]
...        ...                          ...
11359    11360          [Denzel Washington]
11360    11361               [Andrew Erwin]
11361    11362                  [Joel Coen]
11362    11363  [Apichatpong Weerasethakul]
11363    11364              [Clint Bentley]

[11350 rows x 2 columns]


In [87]:
# making it one row per "movie, director" & remove spaces around names
director_data = director_data.explode('director')
director_data['director'] = director_data['director'].str.strip()   
print(director_data)

       movieid                   director
0            1                   Don Hahn
1            2             Hayao Miyazaki
2            3                Jay Russell
3            4                 Steve Carr
4            5       Francis Ford Coppola
...        ...                        ...
11359    11360          Denzel Washington
11360    11361               Andrew Erwin
11361    11362                  Joel Coen
11362    11363  Apichatpong Weerasethakul
11363    11364              Clint Bentley

[11352 rows x 2 columns]


In [88]:
# Removing empty entries and suffix fragments (Jr. etc.)
director_data = director_data[~director_data['director'].isin(['', 'Jr.', 'Sr.', 'II', 'III'])]


In [89]:
# looking up each director's ID: match name in director_data against directorname in the director table
movie_director = director_data.merge(
    director,
    left_on= 'director',
    right_on= 'directorname',
    how='left'
)

In [90]:
# checking the results
movie_director.head(20)

,movieid,director,directorid,directorname
0,1,Don Hahn,1728,Don Hahn
1,2,Hayao Miyazaki,2377,Hayao Miyazaki
2,3,Jay Russell,2726,Jay Russell
3,4,Steve Carr,5571,Steve Carr
4,5,Francis Ford Coppola,2065,Francis Ford Coppola
5,6,Kris Isacsson,3542,Kris Isacsson
6,7,Rodrigo García,5124,Rodrigo García
7,8,Stephan Elliott,5515,Stephan Elliott
8,9,Stewart Raffill,5617,Stewart Raffill
9,10,Andrew Bergman,346,Andrew Bergman


In [91]:
# Counting rows where no ID was found
print(movie_director['directorid'].isna().sum())

0


In [92]:
movie_director.info()
movie_director.isnull().sum().sum()

movie_director = movie_director[['movieid','directorid']].dropna()

movie_director = movie_director.drop_duplicates(
    subset=['movieid', 'directorid']
)
print(movie_director.shape)

<class 'pandas.DataFrame'>
RangeIndex: 11350 entries, 0 to 11349
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   movieid       11350 non-null  int64
 1   director      11350 non-null  str  
 2   directorid    11350 non-null  int64
 3   directorname  11350 non-null  str  
dtypes: int64(2), str(2)
memory usage: 654.1 KB
(11350, 2)


In [93]:
# Export
movie_director = movie_director[['movieid', 'directorid']]
movie_director.to_csv("movie_director.csv", index=False)

In [94]:
# confirm trim happened
movie_director.columns

Index(['movieid', 'directorid'], dtype='str')

In [95]:
# spot check to prove IDs point at right people
check = movie_director.merge(director, on='directorid')
print(check[check['movieid'].isin([1, 7, 3144])])

      movieid  directorid    directorname
0           1        1728        Don Hahn
6           7        5124  Rodrigo García
3141     3144        4210  Michael Landon


# movie_cast

In [96]:
# 1. Load the two files: movie -> cast strings, and the finished cast_member table (with IDs)
junction = movie_lookup
cast_member = pd.read_csv("cast_member_extract.csv")

# 3. Keep only movieid + cast, drop movies without cast info
cast_member_data = junction[['movieid', 'cast']].dropna().copy()

# 4. Repair encoding, then split "A,B,C" into a list and give each actor their own row
cast_member_data['cast'] = cast_member_data['cast'].apply(fix_encoding)   # safe repair, on the whole string
cast_member_data['cast'] = cast_member_data['cast'].str.split(',')
cast_member_data = cast_member_data.explode('cast')
cast_member_data['cast'] = cast_member_data['cast'].str.strip()

# Remove empty entries and leftover suffix fragments like in cast_member (from names like "Bob Smith, Jr.")
cast_member_data = cast_member_data[~cast_member_data['cast'].isin(['', 'Jr.', 'Sr.', 'II', 'III'])]

# Manual fix for the one name that was cut off in the source data
cast_member_data['cast'] = cast_member_data['cast'].replace({'Zsolt PÃ': 'Zsolt Páll'})

print(cast_member_data)

       movieid             cast
0            1     James Levine
0            1     Steve Martin
1            2    Eiko Masuyama
1            2        Gorô Naya
1            2     Ichirô Nagai
...        ...              ...
11363    11364  Martin Bourdieu
11363    11364     Moisés Arias
11363    11364     Molly Parker
11363    11364      Ryan Barber
11363    11364  Vincent Francia

[48296 rows x 2 columns]


In [97]:
# join the two tables
movie_cast = cast_member_data.merge(
    cast_member,
    left_on= 'cast',
    right_on= 'castname',
    how='left'
)

print(movie_cast['castid'].isna().sum())  # check rows with no ID

0


In [98]:
movie_cast.info

<bound method DataFrame.info of        movieid             cast  castid         castname
0            1     James Levine   10374     James Levine
1            1     Steve Martin   22313     Steve Martin
2            2    Eiko Masuyama    6826    Eiko Masuyama
3            2        Gorô Naya    8703        Gorô Naya
4            2     Ichirô Nagai    9630     Ichirô Nagai
...        ...              ...     ...              ...
48291    11364  Martin Bourdieu   15945  Martin Bourdieu
48292    11364     Moisés Arias   17331     Moisés Arias
48293    11364     Molly Parker   17352     Molly Parker
48294    11364      Ryan Barber   20682      Ryan Barber
48295    11364  Vincent Francia   23975  Vincent Francia

[48296 rows x 4 columns]>

In [99]:
movie_cast.info()
movie_cast.isnull().sum().sum()

movie_cast = movie_cast[['movieid','castid']]

movie_cast = movie_cast.drop_duplicates(
    subset=['movieid', 'castid']
)

movie_cast['castid'] = movie_cast['castid'].astype(int) # change type to int

movie_cast.to_csv("movie_cast.csv",index=False)

<class 'pandas.DataFrame'>
RangeIndex: 48296 entries, 0 to 48295
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movieid   48296 non-null  int64
 1   cast      48296 non-null  str  
 2   castid    48296 non-null  int64
 3   castname  48296 non-null  str  
dtypes: int64(2), str(2)
memory usage: 2.7 MB


In [100]:
print(movie_cast.shape)

# spot check. Swap the IDs back into names, so we can see whether they point at the right people
check = movie_cast.merge(cast_member, on='castid') # join on castid: adds the castname that belongs to each ID
print(check[check['movieid'] == 2])                # show only movie 2, then compare with its cast list in movie_lookup.csv

(48284, 2)
    movieid  castid              castname
2         2    6826         Eiko Masuyama
3         2    8703             Gorô Naya
4         2    9630          Ichirô Nagai
5         2   13653     Kiyoshi Kobayashi
6         2   13884        Kôhei Miyauchi
7         2   15336           Makio Inoue
8         2   22443        Sumi Shimamoto
9         2   22623  Tadamichi Tsuneizumi
10        2   22781           Tarô Ishida
11        2   24472          Yasuo Yamada


# reviews

In [101]:
# checking type(s) after warning
user_r['idvscore'].apply(type).value_counts()

idvscore
<class 'str'>      186288
<class 'float'>    133374
Name: count, dtype: int64

In [102]:
print(user_r[user_r['idvscore'].apply(lambda x: isinstance(x, str))]['idvscore'].unique())

['8.0' '9.0' '10.0' '6.0' '7.0' '4.0' '5.0' '3.0' '0.0' '2.0' '1.0' '7'
 '9' '2' '10' '8' '0' '5' '3' '4' '6' '1' ':;;$$'
 ' ayfkm!! wtf!!) Ten reasons I respectfully disagree: (10) Really? Every character in the movie is a caricature. And while I liked me my Beavis & Butt-Head (huh-huh rules) for its then-revelatory reductionist critique of the then-dying music video industr'
 ' cargo carriers'
 'few i think?? i cam out the theather seeing people smile and hearing people say that was worth the waith no bull! I can t wait for a sequel it will be better than this one or just as good as the this one???it just a great winning film in your mind that u'
 ';!?']


In [103]:
broken_mask = user_r['idvscore'].astype(str).str.len() > 5  # scores are short; junk text is long
broken_rows = user_r[broken_mask]
print(len(broken_rows))
broken_rows

3


,url,reviewer,idvscore,source
59309,https://www.metacritic.com/movie/extract,NaN,ayfkm!! wtf!!) Ten reasons I respectfully dis...,user
124298,https://www.metacritic.com/movie/rogue-one-a-s...,storm-troopers,cargo carriers,user
270424,https://www.metacritic.com/movie/underworld-ev...,NaN,few i think?? i cam out the theather seeing pe...,user


In [104]:
# keep only rows where idvscore is short enough to be a real score, not review text
user_r = user_r[user_r['idvscore'].astype(str).str.len() <= 5].copy()

# now safe to convert to numeric
user_r['idvscore'] = pd.to_numeric(user_r['idvscore'], errors='coerce')

print(user_r.shape)
print(user_r['idvscore'].isna().sum())  # should be low/zero now

(316255, 4)
2


In [105]:
# what got removed?
removed = 319662 - 316255
print(removed)

# were there any 4-5 character values that were actually valid and got kept correctly?
# (just confirming nothing weird is left)
print(user_r['idvscore'].describe())

3407
count    316253.000000
mean          6.608184
std           3.185721
min           0.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          10.000000
Name: idvscore, dtype: float64


In [106]:
# dropping the two broken values
user_r = user_r.dropna(subset=['idvscore'])
print(user_r.shape)

(316253, 4)


In [107]:
# checking xpert_r for the same issue
expert_r['idvscore'].apply(type).value_counts()

idvscore
<class 'float'>    238973
Name: count, dtype: int64

In [108]:
# converting idvscore into real numbers and deleting ant row where that wasnt possible
expert_r['idvscore'] = pd.to_numeric(expert_r['idvscore'], errors='coerce')
expert_r = expert_r.dropna(subset=['idvscore'])

In [109]:
# Cleaning up reviewer and rev
reviews = pd.concat([expert_r, user_r], ignore_index=True)
print(reviews.shape)   # expect (555226, 4) roughly

# clean stray quotes and whitespace off reviewer
reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)

reviews[['reviewer']].head(10)

(555224, 4)


,reviewer
0,Andrew O'Hehir
1,A.O. Scott
2,NaN
3,Noel Murray
4,Joshua Rothkopf
5,NaN
6,Bill Goodykoontz
7,Kimberley Jones
8,Peter Travers
9,V.A. Musetto


In [110]:
# where are the text reviews
print(expert_r.columns.tolist())
print(user_r.columns.tolist())

['url', 'reviewer', 'idvscore', 'source']
['url', 'reviewer', 'idvscore', 'source']


In [111]:
# found out the text is still in expert_r.csv
expert_r_raw = pd.read_csv("expert_r.csv")
print(expert_r_raw.columns.tolist())
expert_r_raw.head(3)

['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male', 'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ', 'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual', 'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk', 'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion', 'space', 'time', 'work', 'leisure', 'home', 'money', 'relig', 'death', 'informal', 'swear', 'netspeak', 'assent', 'nonflu', 'filler', 'AllPunc', 'Period', 'Comma', 'Colon', 'SemiC', 'QMark', 'Exclam', 'Dash', 'Quote', 'Apostro', 'Parenth', 'OtherP']


,url,idvscore,reviewer,dateP,Rev,WC,Analytic,Clout,Authentic,Tone,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,https://www.metacritic.com/movie/bronson,100.0,"""Andrew O'Hehir""",None,'Bronson owes a little or a lot to Kubrick s ...,25,73.88,11.52,6.70,25.77,...,4.00,0.0,0.0,0.0,0.0,0.0,0,8.00,0.0,0.0
1,https://www.metacritic.com/movie/bronson,90.0,'A.O. Scott',None,'Bronson invites you to admire its protagonis...,30,13.07,97.69,1.00,99.00,...,6.67,0.0,0.0,0.0,0.0,0.0,0,6.67,0.0,0.0
2,https://www.metacritic.com/movie/bronson,90.0,None,None,'Whether it s Peterson/Bronson s more theatri...,40,72.69,77.33,7.84,2.89,...,7.50,0.0,0.0,0.0,0.0,12.5,0,5.00,0.0,2.5


In [112]:
# and in here
user_r_raw = pd.read_csv("user_r.csv")
print(user_r_raw.columns.tolist())

C:\Users\rolan\AppData\Local\Temp\ipykernel_10660\3861413131.py:2: DtypeWarning: Columns (0: idvscore, 1: thumbsUp, 2: thumbsTot) have mixed types. Specify dtype option on import or set low_memory=False.
  user_r_raw = pd.read_csv("user_r.csv")


['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'thumbsUp', 'thumbsTot', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male', 'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ', 'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual', 'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk', 'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion', 'space', 'time', 'work', 'leisure', 'home', 'money', 'relig', 'death', 'informal', 'swear', 'netspeak', 'assent', 'nonflu', 'filler', 'AllPunc', 'Period', 'Comma', 'Colon', 'SemiC', 'QMark', 'Exclam', 'Dash', 'Quote', 'Apostro', 'Parenth', 'OtherP']


In [113]:
# check for mixed type issue
print(expert_r_raw['idvscore'].apply(type).value_counts())
print(user_r_raw['idvscore'].apply(type).value_counts())

idvscore
<class 'float'>    238973
Name: count, dtype: int64
idvscore
<class 'float'>    279157
<class 'str'>       40331
<class 'int'>         174
Name: count, dtype: int64


In [114]:
# check ints
print(user_r_raw[user_r_raw['idvscore'].apply(lambda x: isinstance(x, int))]['idvscore'].unique())

[10 8 9 7 6 2]


In [115]:
# checking str values
print(user_r_raw[user_r_raw['idvscore'].apply(lambda x: isinstance(x, str))]['idvscore'].unique())

['7' '9' '2' '10' '8' '0' '5' '3' '4' '6' '1' ':;;$$'
 ' ayfkm!! wtf!!) Ten reasons I respectfully disagree: (10) Really? Every character in the movie is a caricature. And while I liked me my Beavis & Butt-Head (huh-huh rules) for its then-revelatory reductionist critique of the then-dying music video industr'
 ' cargo carriers'
 'few i think?? i cam out the theather seeing people smile and hearing people say that was worth the waith no bull! I can t wait for a sequel it will be better than this one or just as good as the this one???it just a great winning film in your mind that u'
 ';!?']


In [116]:
# keep only rows where idvscore is short enough to be a real score, not review text
user_r_raw = user_r_raw[user_r_raw['idvscore'].astype(str).str.len() <= 5].copy()

# convert everything to real numbers
user_r_raw['idvscore'] = pd.to_numeric(user_r_raw['idvscore'], errors='coerce')
user_r_raw = user_r_raw.dropna(subset=['idvscore'])

expert_r_raw['idvscore'] = pd.to_numeric(expert_r_raw['idvscore'], errors='coerce')
expert_r_raw = expert_r_raw.dropna(subset=['idvscore'])

print(user_r_raw.shape)
print(expert_r_raw.shape)

(316253, 100)
(238971, 98)


In [117]:
# adding "source" to differentiate between user and expert later
expert_r_raw['source'] = 'expert'
user_r_raw['source'] = 'user'

reviews = pd.concat([expert_r_raw, user_r_raw], ignore_index=True) # concat stacks them together
print(reviews.shape)

(555224, 101)


In [118]:
# keep only the columns we actually need
reviews = reviews[['url', 'reviewer', 'Rev', 'idvscore', 'source']]

# fixing encoding
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

# clean reviewer: strip stray quotes/whitespace, fix "None" strings, fix encoding
reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)
reviews['reviewer'] = reviews['reviewer'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)

# clean Rev the same way
reviews['Rev'] = reviews['Rev'].astype(str).str.strip().str.strip("'\"")
reviews['Rev'] = reviews['Rev'].replace('None', pd.NA)
reviews['Rev'] = reviews['Rev'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)

print(reviews.shape)
reviews.head(10)

(555224, 5)


,url,reviewer,Rev,idvscore,source
0,https://www.metacritic.com/movie/bronson,Andrew O'Hehir,Bronson owes a little or a lot to Kubrick s Cl...,100.0,expert
1,https://www.metacritic.com/movie/bronson,A.O. Scott,Bronson invites you to admire its protagonist ...,90.0,expert
2,https://www.metacritic.com/movie/bronson,NaN,Whether it s Peterson/Bronson s more theatrica...,90.0,expert
3,https://www.metacritic.com/movie/bronson,Noel Murray,There are two Bronsons on display here: the im...,83.0,expert
4,https://www.metacritic.com/movie/bronson,Joshua Rothkopf,Refn has somehow found his way to an authentic...,80.0,expert
5,https://www.metacritic.com/movie/bronson,NaN,"He s neither victim nor hero, but a man who, i...",80.0,expert
6,https://www.metacritic.com/movie/bronson,Bill Goodykoontz,"This is unhinged genius, an amazing piece of a...",80.0,expert
7,https://www.metacritic.com/movie/bronson,Kimberley Jones,Refn s artful and energetic film never goes fu...,78.0,expert
8,https://www.metacritic.com/movie/bronson,Peter Travers,This movie and Hardy s electrifying performanc...,75.0,expert
9,https://www.metacritic.com/movie/bronson,V.A. Musetto,Tom Hardy gives an amazing performance as Pete...,75.0,expert


In [119]:
# Review text sourced from expert_r.csv/user_r.csv has some apostrophes missing (e.g. 'Kubrick's' → 'Kubrick s'), likely from the original data collection/export process — noted as a known limitation, doesn't affect movieid/reviewer/score accuracy

In [120]:
print(movie_lookup.columns.tolist())
movie_lookup.head(3)

['movieid', 'title', 'awards', 'cast', 'director', 'genre', 'metascore', 'userscore']


,movieid,title,awards,cast,director,genre,metascore,userscore
0,1,Fantasia 2000,#63MostDiscussedMovieof2000,"James Levine,Steve Martin",Don Hahn,"Fantasy,Music,Animation,Family",59,7.2
1,2,Lupin III: The Castle of Cagliostro,#77BestMovieof2000,"Eiko Masuyama,GorÃ´ Naya,IchirÃ´ Nagai,Kiyoshi...",Hayao Miyazaki,"Adventure,Fantasy,Animation,Family",72,8.1
2,3,My Dog Skip,NaN,"Diane Lane,Frankie Muniz,Kevin Bacon",Jay Russell,"Drama,Sport,Family",61,7.9


In [121]:
# use meta.csv to bridge 'reviews' & 'movies' (match url and title)

meta = pd.read_csv("meta.csv", encoding="utf-8")

# step 1: get title from url via meta.csv
reviews = reviews.merge(meta[['url', 'title']], on='url', how='left')
print(reviews['title'].isna().sum())   # how many urls didn't match meta.csv

# step 2: get movieid from title via movie_lookup.csv
reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # how many titles didn't match movie_lookup

0
37


In [122]:
# check 37 titles
unmatched_titles = reviews[reviews['movieid'].isna()]['title'].unique()
print(len(unmatched_titles))
print(unmatched_titles)

2
<ArrowStringArray>
['Cet amour-lÃ ', 'Caterina va in cittÃ ']
Length: 2, dtype: str


In [123]:
# correct broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ ': 'Cet amour-là',
    'Caterina va in cittÃ ': 'Caterina va in città'
})

# defensively clear any leftover movieid columns before merging
reviews = reviews.drop(columns=['movieid', 'movieid_x', 'movieid_y'], errors='ignore')

reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0

37


In [124]:
# reloading meta.vsc and rebuilding reviews from scratch

# reload meta completely fresh, undoing the earlier blanket fix_encoding on title
meta = pd.read_csv("meta.csv", encoding="utf-8")

# rebuild reviews from the raw dataframes (these should still be clean/untouched)
reviews = pd.concat([expert_r_raw, user_r_raw], ignore_index=True)
reviews = reviews[['url', 'reviewer', 'Rev', 'idvscore', 'source']]

reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)
reviews['reviewer'] = reviews['reviewer'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)
reviews['Rev'] = reviews['Rev'].astype(str).str.strip().str.strip("'\"")
reviews['Rev'] = reviews['Rev'].replace('None', pd.NA)

# merge with FRESH meta (title not touched by blanket fix)
reviews = reviews.merge(meta[['url', 'title']], on='url', how='left')

# targeted fix on just the 2 known-broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ ': 'Cet amour-là',
    'Caterina va in cittÃ ': 'Caterina va in città'
})

reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0

37


In [125]:
# locate complete broken titles
raw_title = reviews[reviews['title'].str.contains('amour', na=False)]['title'].iloc[0]
print(repr(raw_title))

raw_title2 = reviews[reviews['title'].str.contains('Caterina', na=False)]['title'].iloc[0]
print(repr(raw_title2))

'Cet amour-lÃ\xa0'
'Caterina va in cittÃ\xa0'


In [126]:
# fixing broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ\xa0': 'Cet amour-là',
    'Caterina va in cittÃ\xa0': 'Caterina va in città'
})
check = reviews[reviews['title'].str.contains('amour|Caterina', na=False)]['title'].unique()
print(check)


<ArrowStringArray>
['Cet amour-là', 'L'amour fou', 'Caterina va in città']
Length: 3, dtype: str


In [127]:
print(reviews.columns.tolist())
print(movie_lookup.columns.tolist())

['url', 'reviewer', 'Rev', 'idvscore', 'source', 'title', 'movieid']
['movieid', 'title', 'awards', 'cast', 'director', 'genre', 'metascore', 'userscore']


In [128]:
# finding unmatched titles again
unmatched_titles = reviews[reviews['movieid'].isna()]['title'].unique()
print(len(unmatched_titles))
print(unmatched_titles)

2
<ArrowStringArray>
['Cet amour-là', 'Caterina va in città']
Length: 2, dtype: str


In [129]:
# check movie lookup csv for same issue with same titles
check2 = movie_lookup[movie_lookup['title'].str.contains('amour|Caterina', na=False)]['title']
print(check2.apply(repr))

1261            'Cet amour-lÃ'
2263    'Caterina va in cittÃ'
5026             "L'amour fou"
Name: title, dtype: str


In [130]:
# fixing it the same way as before

movie_lookup['title'] = movie_lookup['title'].replace({
    'Cet amour-lÃ': 'Cet amour-là',
    'Caterina va in cittÃ': 'Caterina va in città'
})

reviews = reviews.drop(columns=['movieid_x', 'movieid_y', 'movieid'], errors='ignore')
reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0 now

0


In [131]:
# finishing up, according to ERD columns

reviews_final = reviews[['url', 'movieid', 'reviewer', 'Rev', 'idvscore', 'source']].copy()

# rename to match SQL naming convention
reviews_final = reviews_final.rename(columns={
    'reviewer': 'reviewer',
    'Rev': 'review',
    'idvscore': 'idvscore'
})

# generate a review_id, sorted by movieid
reviews_final = reviews_final.sort_values('movieid').reset_index(drop=True)
reviews_final.insert(0, 'reviewid', reviews_final.index + 1)

# dropping the url helper column now that movieid is resolved
reviews_final = reviews_final.drop(columns=['url'])

print(reviews_final.shape)
reviews_final.head(10)

(615866, 6)


,reviewid,movieid,reviewer,review,idvscore,source
0,1,1,Tyranian,Like its predecessor Fantasia 2000 matches som...,6.0,user
1,2,1,Charles Taylor,There s some sort of gross egotism involved in...,30.0,expert
2,3,1,JJH.,"The movie had good animation, I think they sho...",7.0,user
3,4,1,Gemma Files,"Expertly done, and a real joy to watch.",70.0,expert
4,5,1,Bruce Fretts,"Oh well, back to the drawing board.",67.0,expert
5,6,1,F. X. Feeney,NaN,67.0,expert
6,7,1,lasttimeisaw,"Double bill time, FANTASIA is an experimental ...",7.0,user
7,8,1,GagaForGhibli,Fantasia Honestly Is My Favorite Disney Movie!...,9.0,user
8,9,1,Marjorie Baumgarten,Better in theory than in practice.,67.0,expert
9,10,1,Jack Mathews,Whether it s any good depends on your expectat...,63.0,expert


In [132]:
# multiplying userscores by 10

reviews_final.loc[reviews_final['source'] == 'user', 'idvscore'] = reviews_final.loc[reviews_final['source'] == 'user', 'idvscore'] * 10

print(reviews_final.head(10))

   reviewid  movieid             reviewer  \
0         1        1             Tyranian   
1         2        1       Charles Taylor   
2         3        1                 JJH.   
3         4        1          Gemma Files   
4         5        1         Bruce Fretts   
5         6        1         F. X. Feeney   
6         7        1         lasttimeisaw   
7         8        1        GagaForGhibli   
8         9        1  Marjorie Baumgarten   
9        10        1         Jack Mathews   

                                              review  idvscore  source  
0  Like its predecessor Fantasia 2000 matches som...      60.0    user  
1  There s some sort of gross egotism involved in...      30.0  expert  
2  The movie had good animation, I think they sho...      70.0    user  
3            Expertly done, and a real joy to watch.      70.0  expert  
4                Oh well, back to the drawing board.      67.0  expert  
5                                                NaN      67.0  ex

In [133]:
# sanity check
print(reviews_final.groupby('source')['idvscore'].describe())

# NOTE: user scores originally 0-10, expert scores 0-100 (per Metacritic's convention).
# Normalized user scores to match the 0-100 scale for consistency within idvscore.

           count       mean        std  min   25%   50%   75%    max
source                                                              
expert  269221.0  62.697951  20.801788  0.0  50.0  63.0  79.0  100.0
user    346645.0  66.018722  31.806935  0.0  50.0  70.0  90.0  100.0


In [134]:
# final export

reviews_final.to_csv("reviews.csv", index=False)